# Assignment 4

Deadline: 22.04.2026 12:00 CET

- Gallo Alessandro , 25-732-140 , alessandro.gallo2@uzh.ch
- Maruccio Anna , 25-742-800 , anna.maruccio@uzh.ch
- Perbellini Cesare, 25-741-257, cesare.perbellini@uzh.ch
- Venturi Matilde , 25-741-059 , matilde.venturi@uzh.ch

## Prerequisites: Library imports, data load and initialization of the backtest service

In [1]:
# Standard library imports
import os
import sys
import copy
from typing import Optional


# Third party imports
import numpy as np
import pandas as pd

# Add the project root directory to Python path
# Import local modules
project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course', 'src') #changed for Macbook
sys.path.append(project_root)
sys.path.append(src_path)

# Local modules imports
from helper_functions import (
    # load_pickle,
    load_data_spi,
)
from estimation.covariance import Covariance
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.optimization_data import OptimizationData
from optimization.optimization import (
    Optimization,
    Objective,
    MeanVariance,
)
from backtesting.backtest_item_builder.bib_classes import (
    SelectionItemBuilder,
    OptimizationItemBuilder,
)
from backtesting.backtest_item_builder.bibfn_selection import (
    bibfn_selection_min_volume,
    bibfn_selection_gaps,
)
from backtesting.backtest_item_builder.bibfn_optimization_data import (
    bibfn_return_series,
)
from backtesting.backtest_item_builder.bibfn_constraints import (
    bibfn_budget_constraint,
    bibfn_box_constraints,
    bibfn_size_dependent_upper_bounds,
)
from backtesting.backtest_data import BacktestData
from backtesting.backtest_service import BacktestService
from backtesting.backtest import Backtest



In [2]:
PATH_TO_DATA =  '/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/data_backtesting_assignment3/' # <CHANGE THIS TO YOUR PATH TO DATA> 

In [3]:

# Load market and jkp data from parquet files
import pyarrow.parquet as pq

table_market = pq.read_table(f"{PATH_TO_DATA}market_data.parquet")
table_jkp = pq.read_table(f"{PATH_TO_DATA}jkp_data.parquet")

market_data = table_market.to_pandas(strings_to_categorical=False)
jkp_data = table_jkp.to_pandas(strings_to_categorical=False)

# -------------------------
# First, ensure that market data and jkp data 
# have the same dates by forward filling the market data for the missing dates.
# -------------------------

market_data_dates = (
    market_data
    .index.get_level_values('date')
    .unique().sort_values()
)
jkp_data_dates = (
    jkp_data
    .index.get_level_values('date')
    .unique().sort_values()
)

# Find the jkp_data_dates which are not in the market_data_dates
missing_dates = jkp_data_dates[~jkp_data_dates.isin(market_data_dates)]

# Extend the market data for the missing dates using the last available market data (i.e., forward fill).
tmp_dict = {}
for date in missing_dates:
    last_date = market_data_dates[market_data_dates <= date][-1]
    tmp_dict[date] = market_data.loc[last_date]
    
df_missing = pd.concat(tmp_dict, axis=0)
df_missing.index.names = market_data.index.names
market_data_ffill = pd.concat([market_data, df_missing]).sort_index()


# -------------------------
# Define rebalancing dates
# -------------------------

n_month = 3 # We want to rebalance every n_month months

# We want to use the dates from the jkp data for rebalancing, 
# since they are less frequent than the market data dates.
rebdates = (
    jkp_data_dates[
        jkp_data_dates > market_data_dates[0]
    ][::n_month]
    .strftime('%Y-%m-%d').tolist()
)
# Drop the first rebalancing dates which are before 2002-01-01, 
# because of poor data coverage.
rebdates = [date for date in rebdates if date > '2002-01-01']


# -------------------------
# Instantiate the BacktestData class
# and set the market, jkp, and benchmark data as attributes
# -------------------------

data = BacktestData()
data.market_data = market_data_ffill  # notice that we use the forward filled market data here
data.jkp_data = jkp_data
data.bm_series = load_data_spi(path='../data/')

In [4]:
# Define the selection item builders.
selection_item_builders = {
    'gaps': SelectionItemBuilder(
        bibfn=bibfn_selection_gaps,
        width=365 * 3,
        n_days=10,
    ),
    'min_volume': SelectionItemBuilder(
        bibfn=bibfn_selection_min_volume,
        width=365 * 3,
        min_volume=500_000,
        agg_fn=np.median,
    ),
}

# Define the optimization item builders.
optimization_item_builders = {
    'return_series': OptimizationItemBuilder(
        bibfn=bibfn_return_series,
        width=365 * 3,
        fill_value=0,
    ),
    'budget_constraint': OptimizationItemBuilder(
        bibfn=bibfn_budget_constraint,
        budget=1
    ),
    'box_constraints': OptimizationItemBuilder(
        bibfn=bibfn_box_constraints,
        upper=0.1
    ),
    'size_dep_upper_bounds': OptimizationItemBuilder(
        bibfn = bibfn_size_dependent_upper_bounds,
        small_cap = {'threshold': 300_000_000, 'upper': 0.02},
        mid_cap = {'threshold': 1_000_000_000, 'upper': 0.05},
        large_cap = {'threshold': 10_000_000_000, 'upper': 0.1},
    ),
}

# Initialize the backtest service
bs = BacktestService(
    data=data,
    selection_item_builders=selection_item_builders,
    optimization_item_builders=optimization_item_builders,
    rebdates=rebdates,
)

## 1. Maximum Sharpe Ratio Portfolio

a) 

(6 points)

Complete the `MaxSharpe` class below by implementing its methods `set_objective` and `solve`.
The `solve` method should implement an iterative algorithm that quickly approximates the "true" maximimum Sharpe ratio portfolio (given the estimates of mean and covariance). This approximation should be done by repeatedly solving a mean-variance optimization problem, where the risk aversion parameter (which scales the covariance matrix) is adjusted in each iteration. The algorithm should terminate after a maximum of 10 iterations. 

In [5]:
class MaxSharpe(Optimization):

    def __init__(self,
                 constraints: Optional[Constraints] = None,
                 covariance: Optional[Covariance] = None,
                 expected_return: Optional[ExpectedReturn] = None,
                 max_iter: int = 10,
                 tol: float = 1e-6,
                 **kwargs):
        super().__init__(
            constraints=constraints,
            max_iter=max_iter,
            tol=tol,
            **kwargs
        )
        self.covariance = Covariance() if covariance is None else covariance
        self.expected_return = ExpectedReturn() if expected_return is None else expected_return

    def set_objective(self, optimization_data: OptimizationData) -> None:
        X = optimization_data['return_series']
        self.mu = np.asarray(self.expected_return.estimate(X=X, inplace=False), dtype=float).reshape(-1)
        self.Sigma = np.asarray(self.covariance.estimate(X=X, inplace=False), dtype=float)

        self.objective = Objective(
            q=-self.mu,
            P=2 * self.Sigma,
        )
        return None

    def solve(self) -> None:
        gamma = 1.0
        best_sr = -np.inf
        best_weights = None
        best_ret = None
        best_vol = None

        ids = self.constraints.ids

        for _ in range(self.params['max_iter']):
            self.objective = Objective(
                q=-self.mu,
                P=2 * gamma * self.Sigma,
            )

            super().solve()

            if not self.results['status']:
                break

            w_dict = self.results['weights']
            w = np.array([w_dict[i] for i in ids], dtype=float)

            port_ret = float(self.mu @ w)
            port_var = float(w @ self.Sigma @ w)

            if port_var <= 1e-12:
                break

            port_vol = np.sqrt(port_var)
            sharpe = port_ret / port_vol

            if sharpe > best_sr:
                best_sr = sharpe
                best_weights = w_dict.copy()
                best_ret = port_ret
                best_vol = port_vol

            new_gamma = port_ret / (2 * port_var)

            if abs(new_gamma - gamma) < self.params['tol']:
                break

            gamma = new_gamma

        self.results.update({
            'weights': best_weights,
            'status': best_weights is not None,
            'expected_return': best_ret,
            'volatility': best_vol,
            'sharpe_ratio': best_sr,
        })

        return None

b) 

(2 points)

Provide either a theoretical or an empirical justification that your algorithm converges to the true maximum Sharpe ratio portfolio for the given coefficients of mean and covariance.
Hint: If you want to provide an empirical justification, you can perform an optimization for a single point in time by running the following code.

In [6]:
bs.optimization = MaxSharpe(
   covariance=Covariance(method='pearson'),
     expected_return=ExpectedReturn(method='geometric'),
     solver_name='cvxopt',  # <change this to your preferred solver>
     # <optionally add any other arguments you may need, e.g., number of iterations, tolerance, etc.>
)
bs.prepare_rebalancing(rebdates[-1])
bs.optimization.set_objective(bs.optimization_data)
bs.optimization.solve()

bs.optimization.results
sr_ms = bs.optimization.results['sharpe_ratio']
w_ms = bs.optimization.results['weights']

print("MaxSharpe Sharpe ratio:", sr_ms)
gammas = np.logspace(-2, 2, 50)
grid_sharpes = []

cons = bs.optimization.constraints
X = bs.optimization_data['return_series']
asset_names = list(X.columns)

mu = np.asarray(ExpectedReturn(method='geometric').estimate(X=X, inplace=False)).reshape(-1)
Sigma = np.asarray(Covariance(method='pearson').estimate(X=X, inplace=False))

for g in gammas:
    mv = MeanVariance(
        covariance=Covariance(method='pearson'),
        expected_return=ExpectedReturn(method='geometric'),
        constraints=cons,
        risk_aversion=g,
        solver_name='cvxopt',
    )

    mv.set_objective(bs.optimization_data)
    mv.solve()

    raw_w = mv.results['weights']

    if isinstance(raw_w, dict):
        w = np.array([raw_w[a] for a in asset_names], dtype=float)
    else:
        w = np.asarray(raw_w, dtype=float).reshape(-1)

    port_ret = float(mu @ w)
    port_vol = float(np.sqrt(w.T @ Sigma @ w))
    sr = port_ret / port_vol
    grid_sharpes.append(sr)

print("Best Sharpe on MV grid:", max(grid_sharpes))
print("Sharpe from MaxSharpe:", bs.optimization.results['sharpe_ratio'])


MaxSharpe Sharpe ratio: 0.06358081985760669
Best Sharpe on MV grid: 0.06357749348543912
Sharpe from MaxSharpe: 0.06358081985760669


We justify convergence empirically,running the iterative MaxSharpe algorithm for one rebalancing date and compare the resulting Sharpe ratio with the highest Sharpe ratio obtained from a broad grid of standard mean-variance optimizations with different values of the risk-aversion parameter.
In our test, the iterative algorithm returns a Sharpe ratio of 0.0635808, while the best Sharpe ratio found on the mean-variance grid is 0.0635775. The difference found is numerically negligible.
Therefore, the iterative procedure converges to the true maximum Sharpe ratio portfolio (or an extremely close numerical approximation of it) for the given estimates of expected returns and covariance.

## 2. Backtest MaxSharpe with Turnover Penalty

(5 points)

The code below runs a backtest of a MaxSharpe portfolio that includes a turnover penalty. The optimization problem is

$$
\arg\min_{w}\;\Bigl(-\mu^\top w \;+\; \frac{\lambda}{2}\, w^\top \Sigma w \;+\; \tau |w - w^{0}|\Bigr),
$$

where $\mu$ is the vector of expected returns, $\Sigma$ the covariance matrix, and $w^{0}$ the initial portfolio weights.
The parameter $\lambda$ is the risk‑aversion coefficient calibrated in Question 1.a, and $\tau$ is the turnover penalty parameter to be calibrated here.

Your task is to choose a value for the turnover penalty such that the MaxSharpe backtest exhibits an annual turnover of approximately 100\%.

**Hint:** run the backtest for only a few rebalancing dates, compute the resulting turnover using method `bt_ms.turnover`, and iteratively adjust the turnover penalty until the target turnover is reached.


In [ ]:
# Update the backtest service with a MaxSharpe optimization object
bs.optimization = MaxSharpe(
    covariance=Covariance(method='pearson'),
    expected_return=ExpectedReturn(method='geometric'),
    solver_name='cvxopt',     # optionally, change this to your preferred solver
    turnover_penalty=None,    # <your code here>
)

#idea I want bt_ms.turnover to be == 1. create a loop of backtesting initialization in which you change the turnover penalty until condition is not met and print turnove rpenalty 
#how should I update the turnover? 

# Instantiate the backtest object
bt_ms = Backtest()

# Run the backtest
bt_ms.run(bs=bs) 

while(bt_ms.turnover != 1.0):
    bs.optimization['turnover_penalty'] ??

Rebalancing date: 2002-01-31
Rebalancing date: 2002-04-30


KeyboardInterrupt: 

## 4. Simulation and Descriptive Statistics

(3 points)

- Simulate the portfolio returns from your MaxSharpe backtest. Use fixed costs of 1% per annum (p.a.) and variable costs of 0.3% p.a.
- Plot the cumulated returns of the MaxSharpe strategy together with those of the SPI Index.
- Plot the turnover of your MaxSharpe strategy over time.
- Print the annualized turnover (computed as the average turnover over the backtest multiplied by the number of rebalancing per year) for your MaxSharpe strategy.
- Create and print a table with descriptive performance statistics for your MaxSharpe strategy and the SPI Index.


In [ ]:
fixed_costs = 0.01 
variable_costs = 0.003 
return_series = bs.data.get_return_series(weekdays_only=False)

sim_ms = bt_ms.strategy.simulate(
    return_series=return_series,
    fc=fixed_costs,
    vc=variable_costs
)

sim = pd.concat({
    'bm': bs.data.bm_series,
    'ms': sim_ms,
}, axis = 1).dropna()

# Plot the cumulative returns of the strategy and the benchmark

# NOT SURE IS CORRECT- i copied it from backtest 3

np.log((1 + sim)).cumsum().plot(title='Cumulative Performance', figsize = (10, 6))

def sim_outperformance(x: pd.DataFrame, y: pd.Series) -> pd.Series:
    ans = (x.subtract(y, axis=0)).divide(1 + y, axis=0)
    return ans

sim_rel = sim_outperformance(sim, sim['Benchmark'])

np.log((1 + sim_rel)).cumsum().plot(title='Cumulative Out-/Underperformance', figsize = (10, 6))



In [ ]:
# Turnover
to_ms = bt_ms.strategy.turnover(return_series=return_series)
to_ms.plot(title='Turnover', figsize = (10, 6))

In [ ]:

# Annualized turnover per annum (pa) in percentage
to_pa = bt_ms.turnover * 0.01              # <your code here> ?? not sure is per annum 
print(f"The annualized turnover is: {to_pa}%")

In [ ]:
# Decriptive statistics

# <your code here>